In [4]:
# Import PySimpleGUI and PdfReader
import FreeSimpleGUI as sg
from pypdf import PdfReader
import re
from collections import Counter
import spacy

# Define the layout
layout = [
    [sg.Text("PDF Analyzer\n", text_color="black", background_color='light blue')],
    [sg.Text("Select a PDF file:",  text_color="black", background_color='light blue')],
    [sg.Input(key="-FILE-"), sg.FileBrowse(file_types=(("PDF Files", "*.pdf"),))],
    [sg.Button("Read PDF")],
    [sg.Text("", background_color='light blue')],
    [sg.Text("Title:",  key="TITLE", text_color="black", background_color='light blue')],
    [sg.Text("Word Count:",  key="WORD_COUNT", text_color="black", background_color='light blue')],
    [sg.Text("Key Topics:", key="KEY_TOPICS",  text_color="black", background_color='light blue')],
    [sg.Text("Key Names:", key="KEY_NAMES", text_color="black", background_color='light blue')],
    [sg.Text("Reading Level:",  key="READING_LEVEL", text_color="black", background_color='light blue')],
    [sg.Text("", background_color='light blue')],
    [sg.Button("Exit")],
]

# Select Background Color
sg.theme_background_color('light blue')

# Create Window
window = sg.Window("PDF File Selector", layout)

# Get title
def get_title(reader):
    meta = reader.metadata
    title = meta.title
    return title

# Get word count
def get_word_count(reader):
    total_words = 0
    for page in reader.pages:
        text = page.extract_text()
        if text:
            words = text.split()
            total_words += len(words)
            
    return total_words

# Get most common words
def get_most_common_words(reader):
    all_text = ""
    for page in reader.pages:
        all_text += page.extract_text() or ""

    cleaned_text = re.sub(r'\s\w\s', '', all_text)
    cleaned_text = re.sub(r'\s\w\w\s', '', cleaned_text)
    cleaned_text = re.sub(r'\s\w\w\w\s', '', cleaned_text)
    cleaned_text = re.sub(r'\s\w\w\w\w\s', '', cleaned_text)
    words = re.findall(r'\b\w+\b', cleaned_text.lower())
    stop_words = {'the', 'in', 'of', 'and', 'to', 'for', 'is', 'it', 'on', 'with', 'as', 'a', 'i', 't', 's', 'he', 'his', 'her', 'she', 'you', 'me'}
    filtered_words = [w for w in words if w not in stop_words]

    names = get_key_names(reader)
    names = [word.lower() for word in names]
    filtered_names = [w for w in filtered_words if w not in names]
    
    top_words = Counter(filtered_names).most_common(5)
    word_list = []
    for word, count in top_words:
        word_list.append(word)
        
    return word_list

# Get key names
def get_key_names(reader):
    all_text = ""
    for page in reader.pages:
        all_text += page.extract_text()
        
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(all_text)
    names = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    names = Counter(names).most_common(5)
    name_list = []
    for word, count in names:
        name_list.append(word)
    
    return name_list

# Get reading level
def get_reading_level(reader):
    all_text = ""
    for page in reader.pages:
        all_text += page.extract_text()

    words = re.findall(r'\b\w+\b', all_text.lower())
    longest_word = max(words, key=len)
    word_length = len(longest_word)
    
    if word_length < 6:
        return "kindergarden"
    elif word_length < 8:
        return "1st grade"
    elif word_length < 10:
        return "2nd grade"
    elif word_length < 11:
        return "3rd grade"
    elif word_length < 12:
        return "4th grade"
    elif word_length < 13:
        return "5th grade"
    elif word_length < 14:
        return "6th grade"
    elif word_length < 15:
        return "7th grade"
    elif word_length < 16:
        return "8th grade"
    elif word_length < 20:
        return "9th-12th grade"
    else:
        return "college"


# Event loop
while True:
    event, values = window.read()
    
    # Close window if user exits or closes window
    if event in (sg.WIN_CLOSED, "Exit"):
        break
    
     # Process the selected file
    if event == "Read PDF":
        pdf_path = values["-FILE-"]
        if pdf_path:
            reader = PdfReader(pdf_path)
            window["TITLE"].update(f'Title: {get_title(reader)}')
            window["WORD_COUNT"].update(f'Word Count: {get_word_count(reader)}')
            window["KEY_TOPICS"].update(f'Key Topics: {get_most_common_words(reader)}')
            window["KEY_NAMES"].update(f'Key Names: {get_key_names(reader)}')
            window["READING_LEVEL"].update(f'Reading Level: {get_reading_level(reader)}')
        else:
            sg.popup_error("Please select a file first!",  text_color="black", background_color='light blue')

window.close()